In [2]:
import sqlite3

conn = sqlite3.connect('analytics.db')
cur = conn.cursor()

cur.executescript('''
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;

CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    customer_name TEXT,
    region TEXT
);

CREATE TABLE products (
    product_id TEXT PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    unit_price INTEGER
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id TEXT,
    product_id TEXT,
    quantity INTEGER,
    order_date TEXT
);
''')

customers = [
    ('C001','Rahul Sharma','South'),
    ('C002','Priya Singh','North'),
    ('C003','Arjun Kumar','East'),
    ('C004','Sneha Patel','West'),
    ('C005','Vikram Rao','South'),
    ('C006','Anita Desai','North'),
]
products = [
    ('P001','Laptop','Electronics',55000),
    ('P002','Mouse','Electronics',500),
    ('P003','Keyboard','Electronics',1200),
    ('P004','Monitor','Electronics',8000),
    ('P005','Desk Chair','Furniture',4500),
    ('P006','Notebook','Stationery',50),
]
orders = [
    (1,'C001','P001',1,'2024-01-05'),
    (2,'C002','P002',3,'2024-01-06'),
    (3,'C003','P003',2,'2024-01-08'),
    (4,'C004','P004',1,'2024-01-10'),
    (5,'C002','P003',1,'2024-01-11'),
    (6,'C005','P001',1,'2024-01-12'),
    (7,'C001','P002',2,'2024-01-13'),
    (8,'C006','P004',1,'2024-01-14'),
    (9,'C003','P001',1,'2024-01-15'),
    (10,'C002','P004',2,'2024-01-17'),
    (11,'C001','P001',1,'2024-02-02'),
    (12,'C003','P006',5,'2024-02-03'),
    (13,'C005','P005',1,'2024-02-05'),
    (14,'C002','P001',1,'2024-02-08'),
    (15,'C004','P002',4,'2024-02-10'),
    (16,'C001','P004',1,'2024-02-12'),
    (17,'C006','P003',3,'2024-02-14'),
    (18,'C003','P001',1,'2024-03-01'),
    (19,'C005','P002',2,'2024-03-03'),
    (20,'C002','P001',1,'2024-03-05'),
]

cur.executemany('INSERT INTO customers VALUES (?,?,?)', customers)
cur.executemany('INSERT INTO products VALUES (?,?,?,?)', products)
cur.executemany('INSERT INTO orders VALUES (?,?,?,?,?)', orders)

conn.commit()
print("Database ready!")

Database ready!


In [3]:
import pandas as pd

def run_query(sql):
    return pd.read_sql_query(sql, conn)

run_query("SELECT * FROM customers")

,customer_id,customer_name,region
0,C001,Rahul Sharma,South
1,C002,Priya Singh,North
2,C003,Arjun Kumar,East
3,C004,Sneha Patel,West
4,C005,Vikram Rao,South
5,C006,Anita Desai,North


In [4]:
run_query("select * from orders")


,order_id,customer_id,product_id,quantity,order_date
0,1,C001,P001,1,2024-01-05
1,2,C002,P002,3,2024-01-06
2,3,C003,P003,2,2024-01-08
3,4,C004,P004,1,2024-01-10
4,5,C002,P003,1,2024-01-11
5,6,C005,P001,1,2024-01-12
6,7,C001,P002,2,2024-01-13
7,8,C006,P004,1,2024-01-14
8,9,C003,P001,1,2024-01-15
9,10,C002,P004,2,2024-01-17


In [5]:
run_query("select * from products")

,product_id,product_name,category,unit_price
0,P001,Laptop,Electronics,55000
1,P002,Mouse,Electronics,500
2,P003,Keyboard,Electronics,1200
3,P004,Monitor,Electronics,8000
4,P005,Desk Chair,Furniture,4500
5,P006,Notebook,Stationery,50


In [6]:
#warm up 1
run_query('''
select c.customer_name, sum(o.quantity*p.unit_price) as total
from orders o
join products p on o.product_id = p.product_id
join customers c on c.customer_id = o.customer_id
group by c.customer_id, c.customer_name
''')


,customer_name,total
0,Rahul Sharma,119000
1,Priya Singh,128700
2,Arjun Kumar,112650
3,Sneha Patel,10000
4,Vikram Rao,60500
5,Anita Desai,11600


In [28]:
#question 2
run_query('''
with customer_spends as(
select c.customer_name, 
       sum(o.quantity*p.unit_price) as total,
       c.region
from orders o
join products p 
on o.product_id = p.product_id
join customers c 
on c.customer_id = o.customer_id
group by c.customer_id, c.customer_name
),
top_spenders as (
select customer_name,
       total,
       region,
       rank() over (partition by region order by total desc) as rnk
from customer_spends
)
select * from top_spenders
where rnk =1


''')

,customer_name,total,region,rnk
0,Arjun Kumar,112650,East,1
1,Priya Singh,128700,North,1
2,Rahul Sharma,119000,South,1
3,Sneha Patel,10000,West,1


In [7]:
#queston 3-total revenue per product, per month
run_query('''
with revenue_per_product as (
select p.product_id,
       p.product_name,
       strftime('%Y-%m', o.order_date) as month,
       sum(o.quantity*p.unit_price) as total
from orders o
left join products p on o.product_id = p.product_id
group by p.product_id,p.product_name,month
), prev_month_revenue as (

select product_name,
       month,
       total,
       lag(total) over(partition by product_id order by month ) as prev_month_rev
from revenue_per_product
)
select product_name,
       month,
       total,
       prev_month_rev,
       total - prev_month_rev as revenue_change
from prev_month_revenue
''')


,product_name,month,total,prev_month_rev,revenue_change
0,Laptop,2024-01,165000,NaN,NaN
1,Laptop,2024-02,110000,165000.0,-55000.0
2,Laptop,2024-03,110000,110000.0,0.0
3,Mouse,2024-01,2500,NaN,NaN
4,Mouse,2024-02,2000,2500.0,-500.0
5,Mouse,2024-03,1000,2000.0,-1000.0
6,Keyboard,2024-01,3600,NaN,NaN
7,Keyboard,2024-02,3600,3600.0,0.0
8,Monitor,2024-01,32000,NaN,NaN
9,Monitor,2024-02,8000,32000.0,-24000.0


In [8]:
cur.executescript('''
DROP VIEW IF EXISTS customer_public;

CREATE VIEW customer_public AS
SELECT customer_id, region
FROM customers;
''')

print("View created successfully!")

View created successfully!


In [9]:
test_query = "SELECT * FROM customer_public LIMIT 5;"
result = pd.read_sql_query(test_query, conn)
print(result)

  customer_id region
0        C001  South
1        C002  North
2        C003   East
3        C004   West
4        C005  South
